In [1]:
import random
random.seed(42)

import torch

import numpy as np
import pyomo.environ as pyo
import pandas as pd

from pyomo.environ import *
from utils import generate_simple_synthetic_data, const_embed
from trainer import Trainer

np.random.seed(43)

In [2]:

X, y, true_mean, true_std = generate_simple_synthetic_data(n_samples=10)

print(X.head())
print(y.head())


         x1        x2     price
0  0.374540  0.020584  6.506676
1  0.950714  0.969910  2.255445
2  0.731994  0.832443  3.629302
3  0.598658  0.212339  4.297257
4  0.156019  0.181825  5.104630
0    0.064944
1    3.683185
2    2.038461
3    1.054569
4    0.232799
Name: y, dtype: float64


In [3]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

model_config = {
    'hidden_layers': 1,
    'hidden_size': 5,
    'drop': 0.2
    }

training_config = {
    'lr': 0.01,
    'wd': 0.0001,
    'batchsize': 64,
    'iters': 500
    }

trainer = Trainer(X, y, model_config, training_config, device=device)
trainer.create_data_partitions(val_size=0.2, test_size=0.2, random_state=42)

print("X_train shape: ", trainer.X_train.shape)
print("X_val shape: ", trainer.X_val.shape)
print("X_test shape: ", trainer.X_test.shape)

trainer.train()

trainer.test()

X_train shape:  (6, 3)
X_val shape:  (2, 3)
X_test shape:  (2, 3)
Epoch 1/500 - Train Loss: 5.0437, Val Loss: 0.6713
Epoch 2/500 - Train Loss: 4.4072, Val Loss: 0.5139
Epoch 3/500 - Train Loss: 3.4408, Val Loss: 0.3956
Epoch 4/500 - Train Loss: 3.3800, Val Loss: 0.3059
Epoch 5/500 - Train Loss: 2.7077, Val Loss: 0.2313
Epoch 6/500 - Train Loss: 2.5909, Val Loss: 0.1793
Epoch 7/500 - Train Loss: 2.1452, Val Loss: 0.1451
Epoch 8/500 - Train Loss: 1.8272, Val Loss: 0.1251
Epoch 9/500 - Train Loss: 1.5614, Val Loss: 0.1162
Epoch 10/500 - Train Loss: 1.5047, Val Loss: 0.1156
Epoch 11/500 - Train Loss: 1.1716, Val Loss: 0.1198
Epoch 12/500 - Train Loss: 1.2936, Val Loss: 0.1304
Epoch 13/500 - Train Loss: 1.6030, Val Loss: 0.1459
Epoch 14/500 - Train Loss: 1.3279, Val Loss: 0.1649
Epoch 15/500 - Train Loss: 0.8524, Val Loss: 0.1861
Epoch 16/500 - Train Loss: 0.9340, Val Loss: 0.2088
Epoch 17/500 - Train Loss: 0.7136, Val Loss: 0.2321
Epoch 18/500 - Train Loss: 0.7326, Val Loss: 0.2557
Epoch 1

In [4]:
trainer.model

distModel(
  (input_layer): Linear(in_features=3, out_features=5, bias=True)
  (hidden_layers): ModuleList(
    (0): Linear(in_features=5, out_features=5, bias=True)
  )
  (output_mean_layer): Linear(in_features=5, out_features=1, bias=True)
  (output_sd_layer): Linear(in_features=5, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)

In [5]:
cons = trainer.constraint_build()
cons.to_csv('constraints.csv')

In [6]:
cons = pd.read_csv('constraints.csv', index_col=0)
cons

,intercept,layer,node,node_0,node_1,node_2,node_3,node_4
0,-0.094763,0,0,0.253253,-0.214057,-0.066216,NaN,NaN
1,0.165918,0,1,0.292420,-0.622248,0.426366,NaN,NaN
2,-0.049753,0,2,-0.176549,0.232005,0.155098,NaN,NaN
3,0.439311,0,3,-0.098795,0.230881,-0.051803,NaN,NaN
4,0.095011,0,4,0.199581,-0.159095,-0.104677,NaN,NaN
0,0.327901,1,0,-0.080173,-0.186936,0.014291,-0.094443,-0.056829
1,-0.139369,1,1,0.001868,0.168291,0.144993,-0.338088,0.178847
2,0.218551,1,2,0.203920,0.508910,0.368789,-0.322499,-0.348341
3,-0.031785,1,3,-0.143415,0.476409,0.005861,0.287841,-0.134470
4,-0.107298,1,4,0.396933,-0.091936,0.273387,-0.055890,-0.277473


In [7]:
n_sc = 3
contextual_var = trainer.X_test

In [ ]:
model = pyo.ConcreteModel()
    
model.N = pyo.RangeSet(0,0)
model.W = pyo.RangeSet(1,n_sc) # scenarios

pi = [1/n_sc for i in range(n_sc)]

model.x = pyo.Var(contextual_var.columns.sort_values(), model.N)

model.y = pyo.Var(pyo.Any, dense=False, domain=pyo.Reals)

model.rent = pyo.Var(model.W, within=pyo.Reals) # rentabilidad per scenario

#definition of the objective function
def obj_expression(model):
    return sum(pi[w-1] * model.rent[w] for w in model.W)
model.OBJ = pyo.Objective(rule=obj_expression, sense=pyo.maximize)

#definition of the constraints

def rent_expression(model, w):
    return model.rent[w] == sum(model.y['demand', n, w]+model.x['price',n] for n in model.N)
model.rent_expression = pyo.Constraint(model.W, rule=rent_expression)

In [9]:
trainer.y_mean

array([1.19991147])

In [10]:
const_embed(model=trainer, opt_model=model, constaints=cons, outcome='demand', n_scenarios=n_sc)

In [12]:
opt = pyo.SolverFactory('glpk')
results = opt.solve(model, tee=True)

ValueError: Model constraint (rent_expression[1]) contains nonlinear terms that cannot be written to LP format

In [11]:
for c in model.component_objects(Constraint, active=True):
    print(f"Constraint component: {c.name}")
    for index in c:
        print(f"  {index}: {c[index].expr}")


Constraint component: rent_expression
  1: rent[1]  ==  y[demand,0,1]*x[price,0]
  2: rent[2]  ==  y[demand,0,2]*x[price,0]
  3: rent[3]  ==  y[demand,0,3]*x[price,0]
Constraint component: constraint_1_layer0_node0_pred0demand
  None: (x[x1,0] - 0.5198339003650098)/0.30336634092621684*0.2532532 + (x[x2,0] - 0.3911390889126027)/0.22580128160063653*-0.21405725 + (x[price,0] - 3.8124026640068727)/1.414158920256157*-0.066215545 - 0.09476345  <=  v[demand,0,0,0]
Constraint component: constraint_2_layer0_node0_pred0demand
  None: v[demand,0,0,0]  <=  1000.0*v_ind[demand,0,0,0]
Constraint component: constraint_3_layer0_node0_pred0demand
  None: v[demand,0,0,0]  <=  (x[x1,0] - 0.5198339003650098)/0.30336634092621684*0.2532532 + (x[x2,0] - 0.3911390889126027)/0.22580128160063653*-0.21405725 + (x[price,0] - 3.8124026640068727)/1.414158920256157*-0.066215545 - 0.09476345 + 1000.0*(1 - v_ind[demand,0,0,0])
Constraint component: constraint_1_layer0_node1_pred0demand
  None: (x[x1,0] - 0.51983390036